# RG3 gas: all 17 cases, missed-case audit and state-conditioned features
Diagnostic peer comparisons and within-block score percentiles are explanatory only and never predictor inputs.
Feature idea was motivated by observed failures: exploratory repeated-data experiment, not independent validation.
Six features: four signed local-normal jitter departures, normal-jitter support distance, state-reference distance. Reference neighbors drawn only from normal shots in other training blocks; training features are group-cross-fitted.
Context uses preceding10 levels of pressure/backpressure/barrel2/barrel5/hopper; missing history gets no synthetic contextual signal. No LH merge.

In [ ]:
from pathlib import Path
import json
import numpy as np,pandas as pd
from scipy.spatial.distance import cdist
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
nb=json.loads((root/'notebooks/22_rg3_nested_signal_ensemble.ipynb').read_text(encoding='utf-8'))
exec(''.join(nb['cells'][1]['source']).split('scores=[];thresholds=[];foldlogs=[]')[0])
base=pd.read_csv(tables/'rg3_nested_ensemble_predictions.csv',parse_dates=['TimeStamp'])
whole=base.loc[base.scheme.eq('leave_one_block_out')&base.model.eq('max_union')].copy()
density=pd.read_csv(tables/'rg3_density_contrast_predictions.csv',parse_dates=['TimeStamp'])
density=density.loc[density.scheme.eq('leave_one_block_out')&density.model.eq('density_bw2.0')&density.budget.eq(.01),['TimeStamp','prediction']]
audit=whole.merge(d[['TimeStamp']+stdcols],on='TimeStamp',validate='one_to_one').merge(density,on='TimeStamp',validate='one_to_one')
audit['RF_detected']=audit.predicted_train_fpr1
audit['RF_density_detected']=audit.RF_detected|audit.prediction.astype(bool)
audit['margin']=audit.score-audit.threshold
audit['missing_shake_features']=audit[stdcols].isna().sum(axis=1)
audit['normal_percentile_in_block']=np.nan
for block,g in audit.groupby('comparison_block'):
    normals=g.loc[g.actual.eq(0),'score'].to_numpy()
    audit.loc[g.index,'normal_percentile_in_block']=[100*np.mean(normals<=s) for s in g.score]
# Diagnostic same-block peers, not a feature-generation training reference.
rawcols=['Max_Injection_Pressure','Average_Back_Pressure','Cushion_Position','Barrel_Temperature_2','Barrel_Temperature_5','Hopper_Temperature']
peer_rows=[]
for i in np.flatnonzero(y==1):
    peers=np.flatnonzero(groups.eq(groups.iloc[i])&(y==0))
    reference=np.flatnonzero(~groups.eq(groups.iloc[i])&(y==0))
    imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(d.iloc[reference][stdcols])
    sc=RobustScaler().fit(imp.transform(d.iloc[reference][stdcols]))
    q=sc.transform(imp.transform(d.iloc[[i]][stdcols]))
    p=sc.transform(imp.transform(d.iloc[peers][stdcols]))
    distances=cdist(q,p)[0]
    order=np.argsort(distances)[:5]
    for rank,pos in enumerate(order,1):
        z=peers[pos]
        r={'gas_TimeStamp':d.TimeStamp.iloc[i],'normal_TimeStamp':d.TimeStamp.iloc[z],'rank':rank,'shake_distance':float(distances[pos]),'comparison_block':groups.iloc[i]}
        for col in rawcols:r[col+'_gas']=float(d[col].iloc[i]);r[col+'_normal']=float(d[col].iloc[z])
        peer_rows.append(r)
    vals=d.iloc[reference][rawcols]
    scale=(vals.quantile(.75)-vals.quantile(.25)).replace(0,np.nan)
    dif=(d.iloc[i][rawcols].astype(float)-d.iloc[peers[order]][rawcols].median())/scale
    idx=audit.index[audit.TimeStamp.eq(d.TimeStamp.iloc[i])][0]
    audit.loc[idx,'nearest_same_block_normal']=str(d.TimeStamp.iloc[peers[order[0]]])
    audit.loc[idx,'nearest_shake_distance']=float(distances[order[0]])
    audit.loc[idx,'largest_peer_differences_IQR']='; '.join(f'{k}:{dif[k]:+.2f}' for k in dif.abs().nlargest(3).index)
missing=audit.missing_shake_features.gt(0)
audit.loc[missing,'nearest_shake_distance']=np.nan
audit.loc[missing,'nearest_same_block_normal']=''
audit.loc[missing,'largest_peer_differences_IQR']='History missing; imputed peer distance is not evidence of sensor similarity.'
audit.loc[audit.actual.eq(1)].to_csv(tables/'rg3_gas_all17_missed_audit.csv',index=False,encoding='utf-8-sig')
peers_df=pd.DataFrame(peer_rows)
invalid_times=set(audit.loc[missing,'TimeStamp'])
peers_df['valid_shake_history']=~peers_df.gas_TimeStamp.isin(invalid_times)
peers_df.loc[~peers_df.valid_shake_history,'shake_distance']=np.nan
peers_df.to_csv(tables/'rg3_gas_matched_normal_peers.csv',index=False,encoding='utf-8-sig')
for label,mask in [('RF4',audit.RF_detected),('RF_density5',audit.RF_density_detected)]:
    print(label,audit.loc[audit.actual.eq(1)&mask,'TimeStamp'].astype(str).tolist())
print(audit.loc[audit.actual.eq(1),['TimeStamp','score','threshold','margin','missing_shake_features','normal_percentile_in_block']].round(4).to_string(index=False))
# State-conditioned features: use preceding 10-shot levels, reconstructed as
# current minus dev10 reconstructs the prior-10 MEDIAN, not the mean.
# Missing median falls back to current for context matching only.
statecols=['Max_Injection_Pressure','Average_Back_Pressure','Barrel_Temperature_2','Barrel_Temperature_5','Hopper_Temperature']
S=np.column_stack([(d[c]-d[c+'__dev10']).fillna(d[c]).to_numpy(float) for c in statecols])
J=d[stdcols].to_numpy(float)
feature_names=['state_relative_barrel2_shake','state_relative_hopper_shake','state_relative_cushion_shake','state_relative_barrel5_shake','normal_shake_support_distance','state_reference_distance']
fcache={};pcache={}
def contextual(ref,q):
    key=(tuple(ref),tuple(q))
    if key in fcache:return fcache[key]
    assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
    n=ref[y[ref]==0]
    if not len(n):return np.full((len(q),6),np.nan)
    si=SimpleImputer(strategy='median',keep_empty_features=True).fit(S[n])
    ss=RobustScaler().fit(si.transform(S[n]))
    ns=ss.transform(si.transform(S[n]));qs=ss.transform(si.transform(S[q]))
    ji=SimpleImputer(strategy='median',keep_empty_features=True).fit(J[n])
    js=RobustScaler().fit(ji.transform(J[n]))
    nj=js.transform(ji.transform(J[n]));qj=js.transform(ji.transform(J[q]))
    dist=cdist(qs,ns);idx=np.argsort(dist,axis=1)[:,:min(20,len(n))]
    local=nj[idx];med=np.median(local,axis=1)
    mad=np.median(np.abs(local-med[:,None,:]),axis=1)*1.4826
    globalmad=np.median(np.abs(nj-np.median(nj,axis=0)),axis=0)*1.4826
    z=(qj-med)/np.maximum(mad,np.maximum(globalmad*.2,1e-3))
    support=np.sort(cdist(qj,nj),axis=1)[:,:min(5,len(n))].mean(axis=1)
    state_distance=np.take_along_axis(dist,idx,axis=1).mean(axis=1)
    features=np.column_stack([np.clip(z,-25,25),support,state_distance])
    features[~np.isfinite(J[q]).all(axis=1),:]=np.nan
    fcache[key]=features
    return features
def predict(ref,q):
    key=(tuple(ref),tuple(q))
    if key in pcache:return pcache[key]
    if len(np.unique(y[ref]))<2:return np.zeros((len(q),2))
    train=np.full((len(ref),6),np.nan)
    for b in groups.iloc[ref].unique():
        mask=groups.iloc[ref].eq(b).to_numpy()
        if (~mask).any():train[mask]=contextual(ref[~mask],ref[mask])
    test=contextual(ref,q);scores=[]
    for cls in [RandomForestClassifier,ExtraTreesClassifier]:
        m=make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),cls(n_estimators=100,max_depth=5,min_samples_leaf=3,class_weight='balanced',random_state=42,n_jobs=2))
        scores.append(m.fit(train,y[ref]).predict_proba(test)[:,1])
    pcache[key]=np.column_stack([np.mean(scores,axis=0),np.max(scores,axis=0)])
    return pcache[key]
records=[];features_out=[];folds=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,b in enumerate(blocks):
        q=np.flatnonzero(groups.eq(b))
        ref=np.flatnonzero(~groups.eq(b)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
        if len(np.unique(y[ref]))<2:continue
        outer=predict(ref,q);inner=np.full((len(ref),2),np.nan)
        for ib in groups.iloc[ref].unique():
            mask=groups.iloc[ref].eq(ib).to_numpy()
            if (~mask).any():inner[mask]=predict(ref[~mask],ref[mask])
        f=pd.DataFrame(contextual(ref,q),columns=feature_names);f['TimeStamp']=d.TimeStamp.iloc[q].to_numpy();f['scheme']=scheme
        features_out.append(f)
        for k,name in enumerate(['context_RF_ET_mean','context_RF_ET_max']):
            norm=inner[(y[ref]==0)&np.isfinite(inner[:,k]),k]
            cut=float(np.quantile(norm,.99,method='higher')) if len(norm) else 1.
            for l,i in enumerate(q):records.append(dict(scheme=scheme,model=name,TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(outer[l,k]),threshold=cut,prediction=int(outer[l,k]>cut)))
        folds.append(dict(scheme=scheme,block=b,train=len(ref),train_gas=int(y[ref].sum()),test=len(q),test_gas=int(y[q].sum())))
        print('Finished contextual features',scheme,b,flush=True)
pp=pd.DataFrame(records);rows=[]
old=base.loc[base.model.eq('max_union'),['scheme','TimeStamp','predicted_train_fpr1']]
for (scheme,name),g in pp.groupby(['scheme','model'],sort=False):
    z=g.merge(old,on=['scheme','TimeStamp'],validate='one_to_one')
    for combine in ['new_only','OR_RF','AND_RF']:
        pr=z.prediction.astype(bool)
        if combine=='OR_RF':pr=pr|z.predicted_train_fpr1
        if combine=='AND_RF':pr=pr&z.predicted_train_fpr1
        tn,fp,fn,tp=confusion_matrix(z.actual,pr,labels=[0,1]).ravel()
        rows.append(dict(scheme=scheme,model=name,combine=combine,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.,new_model_AP=average_precision_score(z.actual,z.score)))
metrics=pd.DataFrame(rows)
pp.to_csv(tables/'rg3_state_context_predictions.csv',index=False,encoding='utf-8-sig')
metrics.to_csv(tables/'rg3_state_context_metrics.csv',index=False,encoding='utf-8-sig')
pd.concat(features_out,ignore_index=True).to_csv(tables/'rg3_state_context_fold_features.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(folds).to_csv(tables/'rg3_state_context_fold_audit.csv',index=False,encoding='utf-8-sig')
print(metrics.to_string(index=False))


## Executed results
RF4 timestamps: 2020-10-22 04:46:36, 05:10:17; 2020-10-23 05:15:46, 07:15:11. Density adds 2020-10-23 05:59:00.
13 RF misses: none within 0.05 below threshold; 8/13 at or below same-block normal 75th percentile. One has all four prior-10 jitter inputs missing.
Six contextual features tested: both ensembles detect zero gas alone. Whole evaluation RF OR mean remains TP4 FP6; RF OR max TP4 FP7. No promotion.
See reports/rg3_missed_gas_state_context_review.md and all17_missed_audit.csv. Diagnostic percentile/peer differences are not training features.